# Lab 08-01 — Entity/relation graph from passages (the GraphRAG index)

**Track 08 · GraphRAG** — how we index the *structure* of a corpus instead of just its chunks.

Plain RAG embeds chunks and retrieves by vector similarity; GraphRAG changes what gets indexed. Before storing anything, it asks a local LLM to read each passage and pull out the **entities** and the **relations between them** as `(head, relation, tail)` triples, then folds those triples into one graph — nodes are entities, edges carry the relation phrases, and every node remembers which passages mention it. That graph, not a vector index, is what the later labs in this track walk across.

```text
25 passages (rag-mini-wikipedia, deterministic head)
  -> OllamaLLM + json_object per passage (qwen2.5-coder:7b, local)
  -> extract (head, relation, tail) triples
  -> tools/graph build_entity_graph folds them into one networkx graph
  -> graph_stats / top_entities / sample_relations
  -> verification gate (--verify)
```

The graph builder lives in `tools/graph.py` — LangChain has no native entity-graph indexer, so this repo ships its own. Notice what is *missing* from the pipeline: no chunker, no embeddings, no vector store. Extraction is a pure LLM job, and that is the point of the lab.


## Setup

This notebook mirrors `curriculum/08-graphrag/01-entity-graph.py` exactly — the same verified code, split into cells. Two prerequisites must hold before it will run:

- **Ollama serving `qwen2.5-coder:7b` at `localhost:11434`** — the local LLM every extraction call goes to (`llms/ollama.py` talks to it through `langchain-ollama`). Fully local: no API key, no quota. If the server is not up, every triple extraction fails and the graph never materializes.
- **The corpus on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet`, already fetched by the repo's manifest-verified fetchers.

The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. From the terminal the lab runs as:

```bash
python curriculum/08-graphrag/01-entity-graph.py          # run + demo
python curriculum/08-graphrag/01-entity-graph.py --verify # verification gate
```

The next cell installs the lab-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# is a no-op safety net for fresh environments):
#   langchain-ollama -> the Ollama chat backend behind llms/ollama.py
#   pandas           -> read the rag-mini-wikipedia parquet
#   networkx         -> the graph object build_entity_graph returns
%pip install langchain-ollama pandas networkx


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory — this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) — then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd  # noqa: E402

from llms.ollama import OllamaLLM  # noqa: E402
from tools.graph import (  # noqa: E402
    build_entity_graph,
    graph_stats,
    sample_relations,
    top_entities,
)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 25` takes the **deterministic head** of rag-mini-wikipedia — and because every passage costs exactly one LLM extraction call, this number *is* the runtime knob: 25 passages, 25 local calls, no sampling variance between runs. `TOP_K` and `N_SAMPLE_RELATIONS` only shape the demo output (how many hubs to list, how many example triples to show), so they stay small.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
N_PASSAGES = 25  # deterministic head; each passage costs one LLM extraction call
TOP_K = 8  # how many highest-degree entities to print
N_SAMPLE_RELATIONS = 8  # how many example (head, relation, tail) triples to show


## 2. Load — first N passages of rag-mini-wikipedia

`passages.parquet` is a plain table with a `passage` column; `head(n)` keeps the first `n` rows so every run works on the same corpus slice. Each passage is loaded **whole**: there is no chunking here, because the LLM extractor consumes the full text at once. That is the deliberate contrast with every earlier track — the unit of analysis has moved from a chunk to a passage.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N passages of the rag-mini-wikipedia corpus
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts (deterministic head)."""
    df = pd.read_parquet(path)
    return [str(text).strip() for text in df.head(n)["passage"].tolist()]


## 3. Experiment — extract triples per passage and fold them into one graph

This is the GraphRAG index step, in two halves:

- **Extract** — each passage is sent to the local `OllamaLLM` via `json_object` (`qwen2.5-coder:7b` on `localhost:11434`), which returns `(head, relation, tail)` triples. Fully local: no API key, no quota — the cost is wall-clock time, so `build_entity_graph` takes a `progress` callback that prints `extracted done/total passages` on a single line.
- **Fold** — `build_entity_graph` (in `tools/graph.py`) turns every passage's triples into one networkx graph: entities become nodes, relation phrases become edge attributes, and each node remembers which passages mention it.

The result is a single `exp` dict holding the graph, its stats, the top hubs, and sample triples — the artifact labs 03 and 04 build on.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — extract triples per passage and fold them into one graph
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    passages = load_passages(PASSAGES_PATH, N_PASSAGES)
    llm = OllamaLLM()  # local qwen2.5-coder:7b; expose json_object for extraction

    t0 = time.perf_counter()
    graph = build_entity_graph(
        passages,
        llm,
        progress=lambda done, total: print(
            f"  extracted {done}/{total} passages", end="\r", flush=True
        ),
    )
    build_s = time.perf_counter() - t0

    return {
        "passages": passages,
        "graph": graph,
        "build_s": build_s,
        "stats": graph_stats(graph),
        "top": top_entities(graph, TOP_K),
        "examples": sample_relations(graph, N_SAMPLE_RELATIONS),
    }


## 4. Demo

The demo prints the artifact from four angles: the graph's shape (density, average/max degree, connected components), the highest-degree hubs, a handful of extracted triples, and the takeaway. The hubs are the interesting part — a high-degree entity is a *bridge*: the node a multi-hop question has to walk through, which is exactly the structure dense retrieval cannot see.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 08-01 — Entity/relation graph from passages")
    print(f"{len(exp['passages'])} passages, built in {exp['build_s']:.1f}s")
    print("=" * 66)

    s = exp["stats"]
    print(f"\n[1] Graph structure over {s['nodes']} entities / {s['edges']} "
          f"edges:")
    print(f"    density      : {s['density']}")
    print(f"    avg degree   : {s['avg_degree']}")
    print(f"    max degree   : {s['max_degree']}")
    print(f"    components   : {s['connected_components']} "
          f"(largest {s['largest_component']})")

    print(f"\n[2] Highest-degree entities (hubs):")
    for name, degree in exp["top"]:
        print(f"    {degree:3d}  {name}")

    print(f"\n[3] Example extracted triples:")
    for head, relation, tail in exp["examples"]:
        print(f"    {head} -[{relation}]-> {tail}")

    print(f"\n[4] Takeaway")
    print("    The graph is a lossy but *structured* summary of the corpus:")
    print("    entities become nodes and their co-occurring relations become")
    print("    edges. Hubs (high degree) are the entities that connect many")
    print("    others — the bridges a multi-hop question must walk across.")


## 5. Verification gate

The lab ships a `--verify` gate: hard checks the graph must clear — every edge endpoint is a real node, every node remembers its passages, minimum size (≥ 15 entities, ≥ 10 edges), no fully isolated entity graph, and at least one edge carrying a real relation phrase. The gate turns "the lab ran" into "the lab ran *correctly*" — the same discipline every lab in this repo applies.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    s = exp["stats"]
    graph = exp["graph"]

    checks.append(("every edge endpoint is a node",
                   all(graph.has_node(u) and graph.has_node(v)
                       for u, v in graph.edges())))
    checks.append(("every node remembers its passages",
                   all(graph.nodes[n].get("passages") for n in graph.nodes())))
    checks.append((f"graph has >= 15 entities (got {s['nodes']})",
                   s["nodes"] >= 15))
    checks.append((f"graph has >= 10 edges (got {s['edges']})",
                   s["edges"] >= 10))
    checks.append(("avg degree >= 1.0 (no fully isolated entity graph)",
                   s["avg_degree"] >= 1.0))
    checks.append(("at least one edge carries a relation phrase",
                   any(graph[u][v].get("relations") for u, v in graph.edges())))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

25 passages means 25 local LLM extraction calls — expect a few minutes depending on your hardware, with the progress line counting up as `tools/graph` goes. No downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The graph, its hubs, and a sample of the triples — the structured summary the corpus was compressed into.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, the extraction pipeline misbehaved: check that Ollama is serving `qwen2.5-coder:7b` and that the corpus parquet is intact.


In [ ]:
verify_gate(exp)
